In [1]:
import os
import re
from pathlib import Path
import pandas as pd
# import mysql.connector
import pymysql
import subprocess
from datetime import datetime
import warnings
import csv
warnings.filterwarnings("ignore")

In [ ]:
class AnalyseDatabase:
    def __init__(self, path_to_databases):
        
        self.path_to_databases = path_to_databases

    
    def get_connection(self):
        
        connection = pymysql.connect(user="root", password="", host="127.0.0.1", port=3307)
        return connection
    
    def get_schemas(self):
        connection = self.get_connection()
        mySQLcursor = connection.cursor()
        mySQLcursor.execute("show databases")
        mysqlDbs = [x[0] for x in mySQLcursor if x[0]]
        return mysqlDbs

     # remove database mrs from sql file 
    def trim_database(self, database_path_and_filename):
        with open(database_path_and_filename, "r", encoding='ISO-8859-1') as f:
            content = f.read()

        start_text = "USE `mrs`;"
        updated_text = content.split(start_text)[0]

        directory, filename = os.path.split(database_path_and_filename)
        new_filename = f"modified_{filename}"
        file_path = os.path.join(directory, new_filename)

        with open(file_path, "w",encoding='utf-8') as f:
            f.write(updated_text)

        return file_path
    
    
    def get_tables(self):
        connection = self.get_connection()
        mySQLcursor = connection.cursor()
        mySQLcursor.execute("USE mrs")
        mySQLcursor.execute("SHOW TABLES")
        tables = mySQLcursor.fetchall()
        mySqlTables = [x[0] for x in tables]
        return mySqlTables
    
    def get_facility_databases(self): 
        facility_databases = []
        for root, dirs, files in os.walk(self.path_to_databases):
            for file in files:
                if file.endswith(".sql"):
                    file_path = os.path.join(root, file)
                    facility_databases.append(file_path)
        return facility_databases

    def restore_database(self,backup_file):
        backup_file = backup_file.replace("\\", "/")
        # FIRST CHECK FOR EXISTING DATABASES AND DROP THEM
        connection = self.get_connection()

        results = pd.read_sql('SELECT SCHEMA_NAME FROM information_schema.schemata;', con=connection)
        schemas = list(results["SCHEMA_NAME"])
        # Create a cursor object
        cursor = connection.cursor()
        # Execute a query
        cursor.execute('SET foreign_key_checks = 0')
        for schema in schemas:
            if schema in ['client', 'consultation', 'deduplication', 'facility', 'mrs', 'provider', 
                          'report', 'terminology', 'zimepms','repot','maramba','mobile',
                          'reports','mentalhealth','mohcc_sequence']:
                results = cursor.execute(f'DROP DATABASE {schema}')
                print("  >>> DROPPED [{0}]".format(schema))
        cursor.close()
        
        # Force TCP connection to Docker port 3307 with no password
        restore_command = f'mysql --protocol=tcp -h 127.0.0.1 -P 3307 -u root -f < "{backup_file}"'

        try:
            print(f">>> Restoring database: {backup_file}")
            # We removed check=True so it doesn't crash on minor warnings
            # We added capture_output=True so we can read the exact MySQL error
            result = subprocess.run(restore_command, shell=True, capture_output=True, text=True)
            
            if result.returncode != 0:
                # Print the exact error text from the terminal so you can read it
                print(f"!!! Command returned exit status {result.returncode} !!!")
                print(f"ERROR DETAILS: {result.stderr.strip()}")
                
                # Only fail completely if it's an actual authentication or connection rejection
                if "Access denied" in result.stderr or "Can't connect" in result.stderr:
                    return 0
                
                print(">>> Warning detected, but checking for successful import anyway...")
                
            print('>>> DATABASE RESTORE [SUCCESSFUL>>>]')
            return 1
        except Exception as e:
            print(f"Exception: {e}")
            return 0
        


        
    @staticmethod
    def get_facility_details(mapping_file,facility_id):
        facility_name = mapping_file.loc[mapping_file['Facility ID'] == facility_id] ["Facility"].values
        if facility_name.size > 0:
            facility_name = facility_name[0]
            district_name = mapping_file.loc[mapping_file['Facility ID'] == facility_id] ["District"].values
            if district_name.size > 0:
                district_name = district_name[0]
            else:
                district_name = ""
            province_name = mapping_file.loc[mapping_file['Facility ID'] == facility_id] ["Province"].values
            if province_name.size > 0:
                province_name = province_name[0]
            else:
                province_name = ""
        else:
            facility_name = ""
            district_name = ""
            province_name = ""
        return facility_name,district_name,province_name
    
    def get_database_size(self,filename):
        size = round(Path(filename).stat().st_size /(1000*1024),2)
        # size = round(Path(self.path_to_databases + filename).stat().st_size /(1000*1024),2)
        return size
        
    def extracting_data(self,sql,connection):
        try:
            connection = self.get_connection()
            df = pd.read_sql(sql,connection,dtype=str)
            connection.close()
            return df
        except Exception as e:
            print(e)
            return None
    
    @staticmethod
    def concat_dataframes(list_of_dataframes):
        df = pd.concat(list_of_dataframes, ignore_index=True, sort=False)
        return df
    
    @staticmethod
    def get_facility_identifiers(metadata):
        """Returns the siteId from the metadata string."""
        siteId = re.search(r"<string>siteId</string><string>(.*?)</string>", metadata)
        version = re.search(r"<string>version</string><string>(.*?)</string>", metadata)
        last_time_stamp = re.search(r"<string>time</string><string>(.*?)</string>", metadata)
        if siteId:
            return siteId.group(1) , version.group(1) , last_time_stamp.group(1)
        else:
            return None,None,None
    
    def getDbContent(self, filename):
        with open(os.path.join(self.path_to_databases, filename), "r", encoding='ISO-8859-1') as f:
            content = f.read()
        return content
    
    @staticmethod
    def log(processing_time,province="",district="",facility="",site_code="",database_name="",
            number_of_people="",number_of_patients= "",number_of_people_in_art = "", db_size = "", ehr_start_time = "",
              ehr_end_time ="", epms="", datim = "",
            cbs = "",
            status=""
            ): 
         df = pd.DataFrame({"Time File Processed":[processing_time],
                            "province": [province],
                            "district": [district],
                            "facility": [facility],
                            "Site Code":[site_code],
                            "Database Name":[database_name],
                            "Number of People Registered":[number_of_people],
                            "Number of Patients Attended" : [number_of_patients],
                            "Number of People in Art" : [number_of_people_in_art],
                            "Database Size": db_size,
                            "First day EHR used": [ehr_start_time],
                            "Last day EHR used": [ehr_end_time],
                            "Facility with EPMS": [epms],
                            "DATIM File is extractable" : [datim],
                            "CBS File is extractable": [cbs],
                            "Comments":[status]
                            })
         df.to_csv('log.csv', mode='a', index = False, header=None)
    
    
if __name__ == '__main__':
    
    extraction  = AnalyseDatabase(r"/media/tsungie/simba/Q4")
    # extraction  = AnalyseDatabase(r"C:\Users\Simba\Documents\Databases\Cop23\Q4")
    connection = extraction.get_connection()
    facilities = extraction.get_facility_databases()
    mapping_file = pd.read_csv("mapping_file.csv")

    print(">>> Total databases ", len(facilities))

    if not Path("log.csv").exists():
            log_file_header = [
                "Time File Processed",
                "Province",
                "District",
                "Facility",
                "Site Code",
                "Database Name",
                "Number of People Registered",
                "Number of Patients Attended",
                "Number of People in Art" ,
                "Database size",
                "First day EHR used",
                "Last day EHR used",
                "Facility with EPMS",
                "DATIM File is extractable",
                "CBS File is extractable",
                "Comments"
            ]
            with open("log.csv",'w') as file:
                writer = csv.writer(file)
                writer.writerow(log_file_header)
    
    # if Path(extraction.path_to_databases + "log.csv").exists():
    #     df_facilities_= pd.read_csv("log.csv")
    #     ls_facilities_done = list(df_facilities_["Database Name"])
    #     ls_site_codes_done = list(df_facilities_["Site Code"])

    def convert_date(date_string):
            if date_string is None:
                return ""
            date_object = datetime.strptime(date_string, "%Y-%m-%dT%H:%M:%S.%fZ")
            formatted_date = date_object.strftime("%Y-%m-%d")
            formatted_time = date_object.strftime("%H:%M")
            return f"{formatted_date} {formatted_time}"
    
    def extracting_demographics(facility,province_name,district_name,facility_name, facility_id):
        print(">>> working on duplicated art numbers")
        if os.path.exists("Duplicated art numbers.feather"):
            df_art_duplicated = pd.read_feather("Duplicated art numbers.feather")
            df_art_duplicated = df_art_duplicated.drop_duplicates()
        else:
            df_art_duplicated = pd.DataFrame()

        if os.path.exists("Empty art numbers.feather"):
            df_art_empty = pd.read_feather("Empty art numbers.feather")
            df_art_empty = df_art_empty.drop_duplicates()
        else:
            df_art_empty = pd.DataFrame()

        # hts....................................................................................
        if os.path.exists("hts.feather"):
            df_global_hts = pd.read_feather("hts.feather")
            df_global_hts = df_global_hts.drop_duplicates()
        else:
            df_global_hts = pd.DataFrame()

        # cbs ......................................................................................
        if os.path.exists("cbs.feather"):
            df_global_cbs = pd.read_feather("cbs.feather")
            df_global_cbs = df_global_cbs.drop_duplicates()
        else:
            df_global_cbs = pd.DataFrame()

        # hts screening...........................................................................
        if os.path.exists("hts_screening.feather"):
            df_global_hts_screening = pd.read_feather("hts_screening.feather")
            df_global_hts_screening = df_global_hts_screening.drop_duplicates()
        else:
            df_global_hts_screening = pd.DataFrame()
        
        # art.......................................................................................
        if os.path.exists("art.feather"):
            df_global_art = pd.read_feather("art.feather")
            df_global_art = df_global_art.drop_duplicates()
        else:
            df_global_art = pd.DataFrame()

        # df_global_person_investigation..............................................................
        if os.path.exists("person_investigation.feather"):
            df_global_person_investigation = pd.read_feather("person_investigation.feather")
            df_global_person_investigation = df_global_person_investigation.drop_duplicates()
        else:
            df_global_person_investigation = pd.DataFrame()

        # df_global_patient..............................................................
        if os.path.exists("patient.feather"):
            df_global_patient = pd.read_feather("patient.feather")
            df_global_patient = df_global_patient.drop_duplicates()
        else:
            df_global_patient = pd.DataFrame()

         # df_global_epms..............................................................
        if os.path.exists("epms.feather"):
            df_global_epms = pd.read_feather("epms.feather")
            df_global_epms = df_global_epms.drop_duplicates()
        else:
            df_global_epms = pd.DataFrame()

        # df_global_demographics..............................................................
        if os.path.exists("demographics.feather"):
            df_global_demographics = pd.read_feather("demographics.feather")
            df_global_demographics = df_global_demographics.drop_duplicates()
        else:
            df_global_demographics = pd.DataFrame()

        # df_global_art_current_status
        if os.path.exists("art_current_status.feather"):
            df_global_art_current_status = pd.read_feather("art_current_status.feather")
            df_global_art_current_status = df_global_art_current_status.drop_duplicates()
        else:
            df_global_art_current_status = pd.DataFrame()

        # df_global_art_transfer_in
        if os.path.exists("art_transfer_in.feather"):
            df_global_art_transfer_in = pd.read_feather("art_transfer_in.feather")
            df_global_art_transfer_in = df_global_art_transfer_in.drop_duplicates()
        else:
            df_global_art_transfer_in = pd.DataFrame()


        print("   >>> extracting demographics")
        demo_person = extraction.extracting_data("""
                                            SELECT p.person_id ,
                                            p.firstname as FirstName,
                                            p.lastname as LastName,
                                            p.birthdate as Birthdate,
                                            p.sex as Sex,
                                            p.self_identified_gender,
                                            p.nationality,
                                            p.denomination,
                                            p.religion,
                                            p.country_of_birth,
                                            p.marital,
                                            p.city as City,
                                            p.street as Street,
                                            p.town ,
                                            p.education,
                                            p.occupation
                                            FROM client.person p
                                                        """, connection)
        
        try:
            demo_domain_event = extraction.extracting_data("""
                                                SELECT time_stamp as 'Date Rigistered', aggregate_identifier as person_id
                                                            FROM mrs.domain_event_entry
                                                where payload_type like '%PersonRegistered%'
                                                            """, connection)
            demo_domain_event = demo_domain_event.sort_values(by=["person_id","Date Rigistered"])
            demo_domain_event = demo_domain_event.drop_duplicates(subset = ["person_id"],keep = 'first') 
            demo_person = pd.merge(demo_person,demo_domain_event,on = ['person_id'],how = 'left')
        except:
            demo_person =demo_person

        schemas = extraction.get_schemas()
        if 'zimepms' in schemas:
            df_epms = extraction.extracting_data(
                    """
                        select firstname, surname,dateofbirth,dateconfirmedhivpositive,dateofdeath
                        from zimepms.tblpatients

                    """, connection
                )
            df_epms.insert(0,"province_name",province_name)
            df_epms.insert(1,"district_name",district_name)
            df_epms.insert(2,"facility_name",facility_name)
            df_epms.insert(3,"facility_id",facility_id)
            df_epms.insert(4,"db_name",facility.split('Q4')[1])
            df_global_epms = pd.concat([df_global_epms,df_epms])
            # df_global_epms.to_csv("epms.csv")
            df_global_epms.to_feather("epms.feather")

        # Extract all positive cases in EHR - there are three main sources HTS, CBS and ART
        # ART cases can be further split into transfer in and migration
        '''a. extract positives from hts'''
        df_hts = extraction.extracting_data("""
                            SELECT
                                h.time AS Event_date,
                                h.patient_id,
                                h.hts_number,
                                h.laboratory_investigation_id,
                                h.hts_number AS 'HTS Number',
                                h.time AS 'Date of HIV Test',
                                h.purpose AS 'Reason for HIV Test',
                                h.hts_type AS 'Test Method',
                                h.client_already_positive AS 'Already positive',
                                h.client_already_on_art AS 'Already on art',
                                p.result AS HIV_Result
                            FROM consultation.hts h
                            INNER JOIN consultation.person_investigation p ON h.laboratory_investigation_id = p.person_investigation_id
                            WHERE p.result = 'POSITIVE' 
                            """,connection)
        df_hts.insert(0,"province_name",province_name)
        df_hts.insert(1,"district_name",district_name)
        df_hts.insert(2,"facility_name",facility_name)
        df_hts.insert(3,"facility_id",facility_id)
        df_hts.insert(4,"db_name",facility.split('Q4')[1])
        df_global_hts = pd.concat([df_global_hts,df_hts])
        # df_global_hts.to_csv("hts.csv")
        df_global_hts.to_feather("hts.feather")
        
        
        '''b. extract positives from cbs'''
        df_cbs = extraction.extracting_data("""
                            SELECT
                                c.date_of_hiv_test AS 'Event_date',
                                c.person_id,
                                c.date_of_hiv_test AS 'Date of HIV Test',
                                'POSITIVE' AS HIV_Result
                            FROM consultation.cbs c
                        """,connection)
        df_cbs.insert(0,"province_name",province_name)
        df_cbs.insert(1,"district_name",district_name)
        df_cbs.insert(2,"facility_name",facility_name)
        df_cbs.insert(3,"facility_id",facility_id)
        df_cbs.insert(4,"db_name",facility.split('Q4')[1])
        df_global_cbs = pd.concat([df_global_cbs,df_cbs])
        # df_global_cbs.to_csv("cbs.csv")
        df_global_cbs.to_feather("cbs.feather")
        
        df_hts_screening = extraction.extracting_data(
                         """
                            SELECT
                                a.patient_id,
                                a.tested_before,
                                a.art as 'In ART',
                                a.result,
                                a.date_last_tested,
                                a.art_number
                            FROM consultation.hts_screening a
                            where a.result = 'positive'
                        """,connection
        )
        df_hts_screening.insert(0,"province_name",province_name)
        df_hts_screening.insert(1,"district_name",district_name)
        df_hts_screening.insert(2,"facility_name",facility_name)
        df_hts_screening.insert(3,"facility_id",facility_id)
        df_hts_screening.insert(4,"db_name",facility.split('Q4')[1])
        df_global_hts_screening = pd.concat([df_global_hts_screening,df_hts_screening])
        # df_global_hts_screening.to_csv("hts_screening.csv")
        df_global_hts_screening.to_feather("hts_screening.feather")


        df_art = extraction.extracting_data(
                         """
                            SELECT
                                a.date as 'Event_date',
                                a.art_id,
                                a.art_number,
                                a.date_of_hiv_test,
                                a.person_id,
                                a.date_enrolled,
                                a.date,
                                a.art_cohort_number
                            FROM consultation.art a
                        """,connection
        )
        df_art.insert(0,"province_name",province_name)
        df_art.insert(1,"district_name",district_name)
        df_art.insert(2,"facility_name",facility_name)
        df_art.insert(3,"facility_id",facility_id)
        df_art.insert(4,"db_name",facility.split('Q4')[1])
        df_global_art = pd.concat([df_global_art,df_art])
        # df_global_art.to_csv("art.csv")
        df_global_art.to_feather("art.feather")


        df_person_investigation = extraction.extracting_data(
                         """
                            SELECT
                                a.date as 'Event_date',
                                a.person_investigation_id,
                                a.person_id,
                                a.result,
                                a.test
                            FROM consultation.person_investigation a
                            where a.test in ('hiv','HIV DNA PCR','Hiv/Syphilis Duo Test')
                        """,connection
        )
        df_person_investigation.insert(0,"province_name",province_name)
        df_person_investigation.insert(1,"district_name",district_name)
        df_person_investigation.insert(2,"facility_name",facility_name)
        df_person_investigation.insert(3,"facility_id",facility_id)
        df_person_investigation.insert(4,"db_name",facility.split('Q4')[1])
        df_global_person_investigation = pd.concat([df_global_person_investigation,df_person_investigation])
        # df_global_person_investigation.to_csv("person_investigation.csv")
        df_global_person_investigation.to_feather("person_investigation.feather")


        df_patient = extraction.extracting_data(
                        """
                            select patient_id,
                            discharged,person_id,
                            time,discharge_time,
                            back_captured,back_captured_by,
                            date_approved, approved_by
                            from consultation.patient
                        """,connection
        )
        df_patient.insert(0,"province_name",province_name)
        df_patient.insert(1,"district_name",district_name)
        df_patient.insert(2,"facility_name",facility_name)
        df_patient.insert(3,"facility_id",facility_id)
        df_patient.insert(4,"db_name",facility.split('Q4')[1])
        df_global_patient = pd.concat([df_global_patient,df_patient])
        df_global_patient.to_feather("patient.feather")

        # art current status
        df_art_current_status = extraction.extracting_data(
                        """
                            select date,
                            art_id,regimen,
                            state,art_initiation_category
                            from consultation.art_current_status
                        """,connection
        )
        df_art_current_status.insert(0,"province_name",province_name)
        df_art_current_status.insert(1,"district_name",district_name)
        df_art_current_status.insert(2,"facility_name",facility_name)
        df_art_current_status.insert(3,"facility_id",facility_id)
        df_art_current_status.insert(4,"db_name",facility.split('Q4')[1])
        df_global_art_current_status = pd.concat([df_global_art_current_status,df_art_current_status])
        df_global_art_current_status.to_feather("art_current_status.feather")

        # # Select the database
        # cursor.execute("USE consultation")
        # # Execute the SQL command to show tables
        # cursor.execute("SHOW TABLES")
        # # Fetch all the table names
        # tables = cursor.fetchall()


        # if 'art_transfer_in' in tables:
        #     # transfer in
        try:
            df_art_transfer_in = extraction.extracting_data(
                            """
                                select 
                                art_id,date_of_transfer_in,
                                date_of_transfer_out
                                from consultation.art_transfer_in
                            """,connection
            )
            df_art_transfer_in.insert(0,"province_name",province_name)
            df_art_transfer_in.insert(1,"district_name",district_name)
            df_art_transfer_in.insert(2,"facility_name",facility_name)
            df_art_transfer_in.insert(3,"facility_id",facility_id)
            df_art_transfer_in.insert(4,"db_name",facility.split('Q4')[1])
            df_global_art_transfer_in = pd.concat([df_global_art_transfer_in,df_art_transfer_in])
            df_global_art_transfer_in.to_feather("art_transfer_in.feather")
        except:
            print("************** art_transfer_in table not found *************************")
        # else:
        #     print(' **** Art Transfer in table not found')

        
        # Extract duplicated art numbers ........................................................................
        print("extrating duplicated art numbers")
        duplicated_art_numbers = extraction.extracting_data(
            """
                SELECT yt.art_number,c.person_id, c.firstname, c.lastname,c.sex,c.birthdate, yt.date_of_hiv_test, yt.date_enrolled
                FROM consultation.art yt
                JOIN (
                    SELECT art_number
                    FROM consultation.art
                    GROUP BY art_number
                    HAVING COUNT(*) > 1
                ) dup_yt ON yt.art_number = dup_yt.art_number
                JOIN client.person c ON yt.person_id = c.person_id
                where yt.art_number <> ""
                order by yt.art_number
                ; 
            """, connection
        )
        duplicated_art_numbers = pd.merge(duplicated_art_numbers,demo_domain_event, on='person_id', how='left')
        duplicated_art_numbers.insert(0,"province_name",province_name)
        duplicated_art_numbers.insert(1,"district_name",district_name)
        duplicated_art_numbers.insert(2,"facility_name",facility_name)
        duplicated_art_numbers.insert(3,"facility_id",facility_id)
        duplicated_art_numbers.insert(4,"db_name",facility)
        df_art_duplicated = pd.concat([df_art_duplicated,duplicated_art_numbers])
        # df_art_duplicated.to_csv("duplicated_art_numbers.csv")
        df_art_duplicated.to_feather("duplicated_art_numbers.feather")


        # Extract people without art numbers ................................................................
        print("extracting people without art numbers")
        no_art_number = extraction.extracting_data(
            """
                SELECT yt.art_number,c.person_id, c.firstname, c.lastname,c.sex,c.birthdate, yt.date_of_hiv_test, yt.date_enrolled
                FROM consultation.art yt
                JOIN client.person c ON yt.person_id = c.person_id
                where yt.art_number = ""
                order by yt.art_number
            """,connection
        )
        no_art_number = pd.merge(no_art_number,demo_domain_event, on='person_id', how='left')
        no_art_number.insert(0,"province_name",province_name)
        no_art_number.insert(1,"district_name",district_name)
        no_art_number.insert(2,"facility_name",facility_name)
        no_art_number.insert(3,"facility_id",facility_id)
        no_art_number.insert(4,"db_name",facility)
        df_art_empty = pd.concat([df_art_empty,no_art_number])
        # df_art_empty.to_csv("Empty art numbers.csv")
        df_art_empty.to_feather("Empty art numbers.feather")


        
        print("   >>> extracting identification")
        # Identification
        identification = extraction.extracting_data("""
                                                SELECT person_id,
                                                    type as 'identification_type',
                                                    number as 'identification_number' FROM client.identification
                                                    """, connection)
        if identification is not None:
            identification = identification.sort_values(by=["person_id"])
            identification = identification.drop_duplicates(subset=["person_id"], keep='first')
        
        # Phone
        phone = extraction.extracting_data("""
                                                SELECT person_id , number as phone_number FROM client.phone
                                                    """, connection)
        phone = phone.sort_values(by=["person_id"])
        phone = phone.drop_duplicates(subset=["person_id"], keep='first')


        # Art
        art = extraction.extracting_data("""
                                                SELECT person_id, art_number as 'ART Number',date as 'Date of ART Initiation' ,
                                         date_of_hiv_test as 'Date Of HIV Test - From ART',
                                         date_enrolled as 'Date enrolled into ART',date
                                         FROM consultation.art
                                                    """, connection)
        art = art.sort_values(by=["date"])
        art = art.drop_duplicates(subset=["person_id"], keep='first')
        art = art[[
            'person_id', 'ART Number', 'Date Of HIV Test - From ART', 'Date of ART Initiation','Date enrolled into ART'
        ]]
        

        # Patient client profile
        patient_client_profile = extraction.extracting_data( """
                                                        select  person_id ,client_profile, date
                                                        FROM consultation.patient_client_profile
                                                    """, connection)
        if patient_client_profile is not None:
            patient_client_profile = patient_client_profile.sort_values(by=["date"])
            patient_client_profile = patient_client_profile.drop_duplicates(subset=["person_id"], keep='last')

        print("   >>> extracting hts")
        # hts
        hts = extraction.extracting_data( """
                                                        select  p.person_id,
                                                        h.purpose,
                                                        h.entry_point,
                                                        h.date_of_hiv_test as 'Date of HIV test',
                                                        h.hts_number as 'HTS Number',
                                                        pr.result
                                                        FROM consultation.hts h
                                                        left join consultation.patient p
                                                        on h.patient_id = p.patient_id
                                                        left join consultation.person_investigation pr
                                                        on h.laboratory_investigation_id = pr.person_investigation_id
                                                    """, connection)
        if hts is not None:
            hts = hts.sort_values(by=["Date of HIV test"])
            hts = hts.drop_duplicates(subset=["person_id"], keep='last')

        print(">>> merging data")
        demographics_person_identification = pd.merge(demo_person , identification , on = ["person_id"], how = 'left')
        if identification is not None:
            demographics_phone = pd.merge(demographics_person_identification , phone , on =["person_id"],how = 'left')
        else:
            demographics_phone = demographics_person_identification
        demographics_art = pd.merge(demographics_phone , art , on =["person_id"],how = 'left')

        if hts is not None:
            demographics_hts = pd.merge(demographics_art , hts , on =["person_id"],how = 'left')
        else:
            demographics_hts = demographics_art

        if patient_client_profile is not None:
            demographics = pd.merge(demographics_hts, patient_client_profile,on = ["person_id"],how = 'left')
        else:
            demographics = demographics_hts
        
        count = demographics['Date Rigistered'].isna().sum()
        print(">>> Number of records without date of registration ", count)
        demographics = demographics.dropna(subset=['Date Rigistered'])
        demographics['Date Rigistered'] = demographics['Date Rigistered'].apply(convert_date)
        demographics.insert(0,"province_name",province_name)
        demographics.insert(1,"district_name",district_name)
        demographics.insert(2,"facility_name",facility_name)
        demographics.insert(3,"facility_id",facility_id)
        demographics.insert(4,"db_name",facility.split('Q4')[1])
        df_global_demographics = pd.concat([df_global_demographics,demographics])
        print(">>> saving demographics file")
        # df_global_demographics.to_csv("demographics.csv")
        df_global_demographics.to_feather("demographics.feather")

        # print(">>> saving demographics file")
        # facility_file_name = "person_" + facility.split(".sql")[0].split("\\")[-1]
        # demographics.to_csv(facility_file_name + ".csv",index= False)
    
    ls_facilities_done = []
    log_file = pd.read_csv("log.csv")
    ls_facilities_done = list(log_file["Database Name"])
    ls_facilities_done = [element.split("\\")[-1] for element in ls_facilities_done]
    
    count = 0

    cursor = connection.cursor()

    for facility in facilities:
        
        processing_time = datetime.now().strftime("%d/%m/%Y %H:%M:%S")
        db_name = facility.split("\\")[-1]
        db_size = extraction.get_database_size(facility)
        # try:
        count +=1
        
        print(" >>> ", count , "/" , len(facilities), " ",facility)
        print(processing_time, " >>> db size ",db_size)

        if db_name in ls_facilities_done:
            print(">>> facility already done")
            continue

        print(facility)
        
        # trimmed_database = extraction.trim_database(facility)
        return_code = extraction.restore_database(facility)

        if db_size == 0:
            extraction.log(processing_time,"","","","",facility,"","","",db_size,"", "","","","","Corrupt database, 0kb")
            ls_facilities_done.append(db_name)
            continue

        if return_code == 0:
            extraction.log(processing_time,"","","","",facility,"","","",db_size,"","","","","", "Corrupt database, Failed to restore database")
            ls_facilities_done.append(db_name)
            continue
        
        # 1. Check if client, consultation , mrs and reports schema are available
        schemas = extraction.get_schemas()

        # Check if epms exists in database , check if you can  extract reports for cbs and datim
        epms = 'zimepms' in schemas
        if epms:
            epms = "Yes"
        else:
            epms = "No"
        
        cbs = 'consultation' in schemas
        if cbs:
            cbs = "Yes"
        else:
            cbs = "No"
        
        datim = 'report' in schemas
        if datim:
            datim = "Yes"
        else:
            datim = "No"

        all_present = all(item in schemas for item in ['client', 'consultation', 'mrs','report'])
        # Check if the database is complete
        if all_present:
            print(">>> Database is complete")
            df_facility_id = extraction.extracting_data("""
                            SELECT meta_data FROM mrs.domain_event_entry
                            order by time_stamp desc
                            limit 1
                        """, connection)
            # Check if database is empty
            if df_facility_id.empty:
                print(">>> Database is empty")
                extraction.log(processing_time,"","","","",facility,"","","",db_size,"", "",epms, "No","No", "Database is empty")
                ls_facilities_done.append(db_name)
                continue

            content = extraction.getDbContent(facility)
            facility_id,version,last_timestamp = extraction.get_facility_identifiers(df_facility_id.iloc[0, 0])
            first_timestamp = extraction.extracting_data("""
                            SELECT time_stamp FROM mrs.domain_event_entry
                            order by time_stamp asc
                            limit 1
                        """, connection)
            first_timestamp = first_timestamp.iloc[0,0]
            facility_name , district_name, province_name =  extraction.get_facility_details(mapping_file,facility_id)
            number_of_people_registred = extraction.extracting_data("""
                                            select count(*) from client.person
                                                        """, connection)
            
            number_of_patients_attended = extraction.extracting_data("""
                                            select count(*) from consultation.patient
                                                        """, connection)
            
            number_of_people_in_art = extraction.extracting_data("""
                                            select count(*) from consultation.art
                                                        """, connection)
            extracting_demographics(facility,province_name,district_name,facility_name, facility_id)

            extraction.log(processing_time,province_name,district_name,facility_name,facility_id,facility,
                           number_of_people_registred.iloc[0, 0],
                           number_of_patients_attended.iloc[0, 0],
                           number_of_people_in_art.iloc[0, 0],
                           db_size,first_timestamp, last_timestamp,epms, datim,cbs, "Database complete")
            continue

        
        all_present_report = all(item in schemas for item in ['client', 'consultation', 'mrs'])
        if all_present_report:
            mrs_tables = extraction.get_tables()
            if len(mrs_tables) > 4:
                print(">>> Database is incomplete")
                df_facility_id = extraction.extracting_data("""
                                SELECT meta_data FROM mrs.domain_event_entry
                                order by time_stamp desc
                                limit 1
                            """, connection)
                # Check if database is empty
                if df_facility_id.empty:
                    print(">>> Database is empty")
                    extraction.log(processing_time,"","","","",facility,"","","",db_size,"", "",epms, "No","No", "Database is empty")
                    ls_facilities_done.append(db_name)
                    continue

                content = extraction.getDbContent(facility)
                facility_id,version,last_timestamp = extraction.get_facility_identifiers(df_facility_id.iloc[0, 0])
                first_timestamp = extraction.extracting_data("""
                                SELECT time_stamp FROM mrs.domain_event_entry
                                order by time_stamp asc
                                limit 1
                            """, connection)
                first_timestamp = first_timestamp.iloc[0,0]
                facility_name , district_name, province_name =  extraction.get_facility_details(mapping_file,facility_id)
                number_of_people_registred = extraction.extracting_data("""
                                                select count(*) from client.person
                                                            """, connection)
                
                number_of_patients_attended = extraction.extracting_data("""
                                                select count(*) from consultation.patient
                                                            """, connection)
                
                number_of_people_in_art = extraction.extracting_data("""
                                                select count(*) from consultation.art
                                                            """, connection)
                extracting_demographics(facility,province_name,district_name,facility_name, facility_id)

                extraction.log(processing_time,province_name,district_name,facility_name,facility_id,facility,
                            number_of_people_registred.iloc[0, 0],
                            number_of_patients_attended.iloc[0, 0],
                            number_of_people_in_art.iloc[0, 0],
                            db_size,first_timestamp, last_timestamp,epms, datim,cbs, "Database incomplete - missing report schema")
                continue
            else:
                print(">>> Database is incomplete")
                df_facility_id = extraction.extracting_data("""
                                SELECT meta_data FROM mrs.domain_event_entry
                                order by time_stamp desc
                                limit 1
                            """, connection)
                # Check if database is empty
                if df_facility_id.empty:
                    print(">>> Database is empty")
                    extraction.log(processing_time,"","","","",facility,"","","",db_size,"", "",epms, "No","No", "Database is empty")
                    ls_facilities_done.append(db_name)
                    continue

                content = extraction.getDbContent(facility)
                facility_id,version,last_timestamp = extraction.get_facility_identifiers(df_facility_id.iloc[0, 0])
                first_timestamp = extraction.extracting_data("""
                                SELECT time_stamp FROM mrs.domain_event_entry
                                order by time_stamp asc
                                limit 1
                            """, connection)
                first_timestamp = first_timestamp.iloc[0,0]
                facility_name , district_name, province_name =  extraction.get_facility_details(mapping_file,facility_id)
                number_of_people_registred = extraction.extracting_data("""
                                                select count(*) from client.person
                                                            """, connection)
                
                number_of_patients_attended = extraction.extracting_data("""
                                                select count(*) from consultation.patient
                                                            """, connection)
                
                number_of_people_in_art = extraction.extracting_data("""
                                                select count(*) from consultation.art
                                                            """, connection)
                extracting_demographics(facility,province_name,district_name,facility_name, facility_id)

                extraction.log(processing_time,province_name,district_name,facility_name,facility_id,facility,
                            number_of_people_registred.iloc[0, 0],
                            number_of_patients_attended.iloc[0, 0],
                            number_of_people_in_art.iloc[0, 0],
                            db_size,first_timestamp, last_timestamp,epms, datim,cbs, "Database incomplete - incomplete mrs")
                continue
        
        print(">>> Incomplete database")
        extraction.log(processing_time,"","","","",facility,"","","",db_size,"", "",epms, "No","No", "Incomplete database")
    cursor.close()
    connection.close()

>>> Total databases  490
 >>>  1 / 490   /media/tsungie/simba/Q4/bulawayo/Ceshhar101025.sql
23/02/2026 11:24:05  >>> db size  801.76
/media/tsungie/simba/Q4/bulawayo/Ceshhar101025.sql
  >>> DROPPED [client]
  >>> DROPPED [consultation]
  >>> DROPPED [deduplication]
  >>> DROPPED [facility]
  >>> DROPPED [mobile]
  >>> DROPPED [mohcc_sequence]
  >>> DROPPED [mrs]
  >>> DROPPED [provider]
  >>> DROPPED [report]
  >>> DROPPED [terminology]
>>> Restoring database: /media/tsungie/simba/Q4/bulawayo/Ceshhar101025.sql
>>> DATABASE RESTORE [SUCCESSFUL>>>]
>>> Database is complete
>>> working on duplicated art numbers
   >>> extracting demographics
extrating duplicated art numbers
extracting people without art numbers
   >>> extracting identification
   >>> extracting hts
>>> merging data
>>> Number of records without date of registration  0
>>> saving demographics file
 >>>  2 / 490   /media/tsungie/simba/Q4/bulawayo/cowdrayheathcentre151025.sql
23/02/2026 11:24:49  >>> db size  1915.26
/media/

In [ ]:
# import pandas as p


In [ ]:
# df = pd.read_feather("demographics.feather")

In [ ]:
# df['Date Rigistered']

In [ ]:
df.shape

In [ ]:
df.to_csv("demos.csv")